# 🎮 AI Readiness Simulation Game

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CrashingGuru/ITUAIReadiness/blob/main/simulation/AI_Readiness_Game_Colab.ipynb)

An interactive simulation based on the **ITU AI Readiness Framework 2025** (Report 2.0).

Country delegates describe their AI readiness across 13 dimensions, and 6 AI agents analyze, score, and simulate "what-if" policy scenarios using a knowledge base of 57+ national AI strategy documents.

---

### How to use this notebook

1. **Run cells 1-5 in order** (first-time setup, ~20 min)
2. **Play the game** from Cell 6 onwards
3. To persist data across sessions, enable Google Drive in Cell 2

### Requirements

| Colab Tier | LLM Model | RAM | Notes |
|-----------|-----------|-----|-------|
| **Free** | `llama3.1:8b` | 12 GB | Works but slower, smaller context |
| **Pro** | `qwen2.5:14b` | 50+ GB | Recommended — better reasoning |

## Cell 1 — Install Ollama & Pull Models

Installs the Ollama LLM server, starts it, and pulls the required models.

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in background
import subprocess, time
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(5)
print("Ollama server started.")

# ============================================================
# Choose your model (uncomment ONE line):
# ============================================================
MODEL = "llama3.1:8b"       # Free tier (12GB RAM) — safe default
# MODEL = "qwen2.5:14b"     # Pro tier (16GB+ RAM) — better reasoning
# ============================================================

print(f"Pulling LLM model: {MODEL} ...")
!ollama pull {MODEL}

print("Pulling embedding model: nomic-embed-text ...")
!ollama pull nomic-embed-text

print("\n✅ Models ready!")
!ollama list

## Cell 2 — Clone Repository & Install Dependencies

In [ ]:
# ============================================================
# Optional: Mount Google Drive to persist the Knowledge Base
# Uncomment the next 3 lines to enable persistence across sessions
# ============================================================
# from google.colab import drive
# drive.mount('/content/drive')
# PERSIST_TO_DRIVE = True
PERSIST_TO_DRIVE = False

# Clone the repository
import os
if not os.path.exists("ai-readiness-game"):
    !git clone https://github.com/CrashingGuru/ITUAIReadiness.git ai-readiness-game
    print("Repository cloned.")
else:
    print("Repository already exists, pulling latest...")
    !cd ai-readiness-game && git pull

%cd ai-readiness-game/simulation

# Install Python dependencies
!pip install -q ollama chromadb fastapi uvicorn[standard] websockets \
    sqlmodel httpx rich networkx PyMuPDF pydantic-settings python-multipart

print("\n✅ Dependencies installed!")

## Cell 3 — Configure & Ingest Documents into Knowledge Base

Parses all 57 documents (PDFs + text files) from `InputDocs/`, chunks them, embeds with `nomic-embed-text`, and stores in ChromaDB.

**First run takes ~15-20 minutes.** If you enabled Google Drive persistence, subsequent runs skip this.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, '.')

# Override config for Colab
from server.config import settings
settings.llm_model = MODEL
settings.ollama_base_url = "http://localhost:11434"

# If persisting to Google Drive, redirect ChromaDB storage
if PERSIST_TO_DRIVE:
    drive_path = Path("/content/drive/MyDrive/ai-readiness-game")
    drive_path.mkdir(parents=True, exist_ok=True)
    settings.chromadb_dir = drive_path / "chromadb"
    settings.db_path = drive_path / "game.db"
    print(f"Data will be persisted to Google Drive: {drive_path}")

# Check if KB already has data (skip ingestion if so)
from server.knowledge.kb import KnowledgeBase
kb = KnowledgeBase()
stats = kb.stats()
total_docs = sum(stats.values()) if isinstance(stats, dict) else 0

if total_docs > 100:
    print(f"Knowledge Base already has {total_docs} documents. Skipping ingestion.")
    print(f"To re-ingest, run: !python -m server.knowledge.ingest --reset")
else:
    print("Ingesting documents into Knowledge Base...")
    print("This takes ~15-20 minutes on first run.\n")
    !python -m server.knowledge.ingest --reset
    print("\n✅ Knowledge Base ready!")

## Cell 4 — Start the FastAPI Server

In [ ]:
import subprocess, sys, time

# Write a small launcher that sets the model before starting
launcher = f"""
import sys, os
sys.path.insert(0, '.')
os.environ['AIREADY_LLM_MODEL'] = '{MODEL}'
import uvicorn
from server.config import settings
uvicorn.run("server.main:app", host="0.0.0.0", port=8000, reload=False)
"""
with open("_colab_server.py", "w") as f:
    f.write(launcher)

server_proc = subprocess.Popen(
    [sys.executable, "_colab_server.py"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(3)

# Verify server is running
import httpx
try:
    r = httpx.get("http://localhost:8000/health", timeout=5)
    print("✅ Server is running on port 8000")
except:
    # Try the root endpoint
    try:
        r = httpx.get("http://localhost:8000/", timeout=5)
        print("✅ Server is running on port 8000")
    except:
        print("⚠️  Server may still be starting. Wait a few seconds and run the next cell.")

print(f"Using LLM model: {MODEL}")
print("API docs: http://localhost:8000/docs (not accessible externally on Colab)")

## Cell 5 — Game API Functions

These helper functions replace the CLI. Use them in subsequent cells to play the game.

In [ ]:
import httpx, json
from IPython.display import display, Markdown, HTML

BASE = "http://localhost:8000"
client = httpx.Client(timeout=600)

# Current session state
_session = {"id": None, "country": None, "name": None}


def register(country: str, delegate_name: str) -> str:
    """Register as a country delegate. Returns session ID."""
    r = client.post(f"{BASE}/delegates/register",
                    json={"country": country, "delegate_name": delegate_name})
    r.raise_for_status()
    data = r.json()
    _session["id"] = data["session_id"]
    _session["country"] = country
    _session["name"] = delegate_name
    display(Markdown(f"### ✅ Registered\n- **Country:** {country}\n- **Delegate:** {delegate_name}\n- **Session:** `{data['session_id']}`"))
    return data["session_id"]


def assess(description: str) -> dict:
    """Submit a country assessment (free-form text)."""
    sid = _session["id"]
    if not sid:
        print("❌ Register first with register('Country', 'Name')")
        return {}
    r = client.post(f"{BASE}/delegates/{sid}/scenario",
                    json={"input_text": description, "input_method": "narrative"})
    r.raise_for_status()
    data = r.json()

    # Display scores as a table
    scores = data.get("scores", [])
    if scores:
        table = "| Dimension | Score | Confidence |\n|-----------|-------|------------|\n"
        for s in scores:
            table += f"| D{s['dimension_id']}: {s.get('dimension_name','')} | {s['score']:.1f}/5.0 | {s.get('confidence',0):.0%} |\n"
        composite = data.get("composite_score", 0)
        table += f"\n**Composite Score: {composite:.2f} / 5.0**"
        display(Markdown(f"### 📊 Assessment Results for {_session['country']}\n\n{table}"))

    narrative = data.get("agent_narrative", "")
    if narrative:
        display(Markdown(f"**Agent Analysis:** {narrative}"))

    return data


def clarify(question: str) -> dict:
    """Ask a clarification about the ITU AI Readiness Framework."""
    sid = _session["id"]
    if not sid:
        print("❌ Register first with register('Country', 'Name')")
        return {}
    r = client.post(f"{BASE}/game/{sid}/clarify",
                    json={"question": question})
    r.raise_for_status()
    data = r.json()

    answer = data.get("answer", "No answer")
    dims = data.get("related_dimensions", [])
    agent = data.get("agent", "unknown")

    display(Markdown(
        f"### 💡 Clarification\n\n"
        f"**Q:** {question}\n\n"
        f"**A:** {answer}\n\n"
        f"*Related Dimensions: {dims} | Agent: {agent}*"
    ))
    return data


def whatif(question: str) -> dict:
    """Run a what-if policy simulation."""
    sid = _session["id"]
    if not sid:
        print("❌ Register first with register('Country', 'Name')")
        return {}
    r = client.post(f"{BASE}/game/{sid}/whatif",
                    json={"question": question})
    r.raise_for_status()
    data = r.json()

    narrative = data.get("agent_narrative", data.get("narrative", ""))
    effects = data.get("causal_effects", [])

    md = f"### 🔮 What-If Analysis\n\n**Scenario:** {question}\n\n"
    if narrative:
        md += f"**Analysis:** {narrative}\n\n"
    if effects:
        md += "**Causal Effects:**\n\n| Dimension | Effect |\n|-----------|--------|\n"
        for e in effects:
            md += f"| D{e.get('dimension_id', '?')} | {e.get('effect', 0):+.2f} |\n"

    display(Markdown(md))
    return data


def decide(decision: str) -> dict:
    """Record a policy decision."""
    sid = _session["id"]
    if not sid:
        print("❌ Register first with register('Country', 'Name')")
        return {}
    r = client.post(f"{BASE}/game/{sid}/decide",
                    json={"decision_text": decision})
    r.raise_for_status()
    data = r.json()
    display(Markdown(f"### ✅ Decision Recorded\n\n**Decision:** {decision}"))
    return data


def dashboard() -> dict:
    """View current scores and session status."""
    sid = _session["id"]
    if not sid:
        print("❌ Register first with register('Country', 'Name')")
        return {}
    r = client.get(f"{BASE}/dashboard/{sid}")
    r.raise_for_status()
    data = r.json()

    scores = data.get("scores", [])
    if scores:
        table = "| Dimension | Score | Confidence |\n|-----------|-------|------------|\n"
        for s in scores:
            table += f"| D{s['dimension_id']}: {s.get('dimension_name','')} | {s['score']:.1f}/5.0 | {s.get('confidence',0):.0%} |\n"
        display(Markdown(f"### 📊 Dashboard — {_session['country']}\n\n{table}"))
    else:
        display(Markdown(f"### 📊 Dashboard — {_session['country']}\n\nNo scores yet. Run `assess()` first."))

    return data


def impact() -> dict:
    """View the causal graph relationships between dimensions."""
    r = client.get(f"{BASE}/dashboard/causal-graph")
    r.raise_for_status()
    data = r.json()

    edges = data.get("edges", [])
    if edges:
        table = "| From | To | Weight |\n|------|-----|--------|\n"
        for e in edges:
            table += f"| D{e['from']} | D{e['to']} | {e['weight']:.2f} |\n"
        display(Markdown(f"### 🔗 Causal Graph (Dimension Interdependencies)\n\n{table}"))

    return data


def agents() -> list:
    """List all 6 AI agents and their dimension coverage."""
    r = client.get(f"{BASE}/dashboard/agents")
    r.raise_for_status()
    data = r.json()

    table = "| Agent | Dimensions | Description |\n|-------|-----------|-------------|\n"
    for a in data:
        table += f"| {a['name']} | {a['dimensions']} | {a['description']} |\n"
    display(Markdown(f"### 🤖 AI Agents\n\n{table}"))

    return data


def kb_stats() -> dict:
    """View Knowledge Base statistics."""
    r = client.get(f"{BASE}/dashboard/kb-stats")
    r.raise_for_status()
    data = r.json()
    display(Markdown(f"### 📚 Knowledge Base Stats\n\n```json\n{json.dumps(data, indent=2)}\n```"))
    return data


print("✅ Game functions loaded!")
print()
print("Available commands:")
print("  register('Country', 'Name')  — Register as a delegate")
print("  assess('description...')      — Submit country assessment")
print("  clarify('question...')        — Ask about the framework")
print("  whatif('scenario...')          — Simulate a policy change")
print("  decide('decision...')          — Record a policy decision")
print("  dashboard()                    — View current scores")
print("  impact()                       — View causal graph")
print("  agents()                       — List AI agents")
print("  kb_stats()                     — Knowledge Base stats")

---

## 🎮 Play the Game

Run the cells below to interact with the simulation. Edit and re-run as needed.

### Step 1 — Register as a delegate

In [ ]:
sid = register("Ethiopia", "delegate1")

### Step 2 — Ask clarifications about the framework

In [ ]:
clarify("What is the data marketplace dimension?")

In [ ]:
clarify("How does digital infrastructure affect AI readiness?")

### Step 3 — Submit a country assessment

In [ ]:
assess("""
Ethiopia is in the early stages of AI development. The country has a growing
tech ecosystem centered around Addis Ababa, with initiatives like the Ethiopian
AI Institute. Internet connectivity remains limited in rural areas (~25% penetration).
There is a national digital transformation strategy but no dedicated AI strategy yet.
Several universities offer AI/ML courses. Open data availability is limited.
The government has expressed interest in using AI for agriculture and healthcare.
""")

### Step 4 — Run what-if scenarios

In [ ]:
whatif("What if Ethiopia invests $100M in national AI compute infrastructure and 5G deployment?")

In [ ]:
whatif("What if Ethiopia launches a national open data portal with 500 government datasets?")

### Step 5 — Make decisions

In [ ]:
decide("Ethiopia will launch a national open data portal by 2027 with 500+ government datasets.")

### Step 6 — View dashboard and stats

In [ ]:
dashboard()

In [ ]:
agents()

In [ ]:
impact()

In [ ]:
kb_stats()

---

## 🔧 Utilities

In [ ]:
# Stop the server (run this when done)
server_proc.terminate()
print("Server stopped.")

In [ ]:
# Re-ingest documents (if you added new files to InputDocs/)
!python -m server.knowledge.ingest --reset